# 최종 활용 적합성 분석

## 분석 목적

앞선 EDA에서 도출한 지역수요와 유휴재산 조건을 결합하여
개별 유휴 공유재산별 추천 활용유형과 우선 검토 수준을 정리한다.

최종 결과는 특정 시설의 설치 가능성을 확정하는 것이 아니라,

**지역의 필요 × 재산의 조건**

을 기준으로 향후 법적·물리적·사업성 검토가 우선적으로 필요한
유휴 공유재산을 제시하는 것을 목적으로 한다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np

BASE_PATH = "/content/drive/MyDrive/부산_BigData_공모전/"

property_final = pd.read_csv(
    BASE_PATH + "03_analysis_output/유휴공유재산_개별활용유형추천_1차.csv",
    encoding="utf-8-sig"
)

primary = pd.read_csv(
    BASE_PATH + "03_analysis_output/유휴공유재산_A군_186건.csv",
    encoding="utf-8-sig"
)

priority = pd.read_csv(
    BASE_PATH + "03_analysis_output/활용유형별_A군_우선검토순위.csv",
    encoding="utf-8-sig"
)

print("[개별 활용유형 추천]")
print(property_final.shape)

print("\n[A군]")
print(primary.shape)

print("\n[활용유형별 A군 순위]")
print(priority.shape)

print("\nA군 고유 주소:", primary["lctnAddr"].nunique())
print("순위 데이터 고유 주소:", priority["lctnAddr"].nunique())

In [ ]:
# 활용유형별 우선검토 정보를 재산 단위로 변환
priority_wide = (
    priority.pivot_table(
        index="lctnAddr",
        columns="검토활용유형",
        values="우선검토균형도",
        aggfunc="max"
    )
    .reset_index()
)

priority_wide.columns.name = None

priority_wide = priority_wide.rename(
    columns={
        "아동·가족 생활지원형": "아동_우선검토균형도",
        "고령 생활지원형": "고령_우선검토균형도",
        "활동·교류 지원형": "활동교류_우선검토균형도"
    }
)

print(priority_wide.shape)
display(priority_wide.head())

In [ ]:
# A군 186건에 활용유형별 결과 결합
final_candidates = primary.merge(
    priority_wide,
    on="lctnAddr",
    how="left",
    validate="one_to_one"
)

final_cols = [
    "gugun",
    "행정동명",
    "lctnAddr",
    "zoneArea",
    "land_type_clean",
    "대표추천유형",
    "추천가능유형",
    "추천가능유형수",
    "최고_수요균형도",
    "면적_pct",
    "재산검토그룹",
    "아동·가족 생활지원형",
    "고령 생활지원형",
    "활동·교류 지원형",
    "아동_우선검토균형도",
    "고령_우선검토균형도",
    "활동교류_우선검토균형도",
    "x",
    "y"
]

final_candidates = final_candidates[
    [c for c in final_cols if c in final_candidates.columns]
].copy()

print("최종 shape:", final_candidates.shape)
print("고유 주소:", final_candidates["lctnAddr"].nunique())
print("주소 중복:", final_candidates["lctnAddr"].duplicated().sum())

print("\n[대표추천유형]")
print(final_candidates["대표추천유형"].value_counts())

display(final_candidates.head())

In [ ]:
# 부산 206개 행정동 지역수요 프로필 로드
profile = pd.read_csv(
    BASE_PATH + "02_processed_data/지역수요프로필_행정동별.csv",
    encoding="utf-8-sig"
)

print("지역수요 프로필:", profile.shape)
print("행정동 중복:", profile["행정동명"].duplicated().sum())

# 부산 전체 기준 구성비 계산
city_under20_share = (
    profile["20대미만_활동인구"].sum()
    / profile["연령합계_전체활동인구"].sum()
)

city_over60_share = (
    profile["60대이상_활동인구"].sum()
    / profile["연령합계_전체활동인구"].sum()
)

city_workplace_share = (
    profile["평균직장인구수"].sum()
    / profile["전체활동인구"].sum()
)

city_visit_share = (
    profile["평균방문인구수"].sum()
    / profile["전체활동인구"].sum()
)

# 설명용 LQ
profile["20대미만_LQ"] = (
    profile["20대미만_구성비"] / city_under20_share
)

profile["60대이상_LQ"] = (
    profile["60대이상_구성비"] / city_over60_share
)

profile["직장_LQ"] = (
    profile["직장인구_구성비"] / city_workplace_share
)

profile["방문_LQ"] = (
    profile["방문인구_구성비"] / city_visit_share
)

profile["활동교류_LQ"] = profile[
    ["직장_LQ", "방문_LQ"]
].max(axis=1)

demand_detail_cols = [
    "행정동명",
    "전체활동인구",
    "20대미만_활동인구",
    "20대미만_구성비",
    "20대미만_LQ",
    "60대이상_활동인구",
    "60대이상_구성비",
    "60대이상_LQ",
    "아동돌봄시설_전체",
    "고령층생활서비스_전체",
    "직장인구_구성비",
    "방문인구_구성비",
    "직장_LQ",
    "방문_LQ",
    "활동교류_LQ"
]

final_candidates = final_candidates.merge(
    profile[demand_detail_cols],
    on="행정동명",
    how="left",
    validate="many_to_one"
)

print("결합 후:", final_candidates.shape)

print(
    "지역수요 결합 실패:",
    final_candidates["전체활동인구"].isna().sum()
)

In [ ]:
# 면적 percentile을 읽기 쉬운 표현으로 변환
final_candidates["면적_상위비율"] = (
    (1 - final_candidates["면적_pct"]) * 100
)

def make_reason(row):

    area_text = (
        f"대 지목 {row['zoneArea']:,.1f}㎡, "
        f"전체 유휴재산 중 면적 상위 약 {row['면적_상위비율']:.1f}% 수준"
    )

    if row["대표추천유형"] == "아동·가족 생활지원형":
        return (
            f"20대 미만 활동인구 {row['20대미만_활동인구']:,.1f}명, "
            f"구성비 {row['20대미만_구성비']*100:.1f}% "
            f"(부산 평균 대비 LQ {row['20대미만_LQ']:.2f}). "
            f"{area_text}."
        )

    elif row["대표추천유형"] == "고령 생활지원형":
        return (
            f"60대 이상 활동인구 {row['60대이상_활동인구']:,.1f}명, "
            f"구성비 {row['60대이상_구성비']*100:.1f}% "
            f"(부산 평균 대비 LQ {row['60대이상_LQ']:.2f}). "
            f"{area_text}."
        )

    else:
        dominant = (
            "직장활동"
            if row["직장_LQ"] >= row["방문_LQ"]
            else "방문활동"
        )

        return (
            f"전체 활동인구 {row['전체활동인구']:,.1f}명이며 "
            f"{dominant} 특화도가 높음 "
            f"(직장 LQ {row['직장_LQ']:.2f}, "
            f"방문 LQ {row['방문_LQ']:.2f}). "
            f"{area_text}."
        )


final_candidates["추천근거"] = final_candidates.apply(
    make_reason,
    axis=1
)

final_candidates["검토유의사항"] = (
    "현재 분석은 지역수요·지목·면적을 이용한 1차 우선 검토 결과이며, "
    "실제 활용 전 용도지역·접도·필지형상·현황·관련 법규·사업성 등의 추가 검토가 필요함."
)

display(
    final_candidates[
        [
            "대표추천유형",
            "행정동명",
            "lctnAddr",
            "zoneArea",
            "추천근거"
        ]
    ].head(10)
)

In [ ]:
final_output_cols = [
    "gugun",
    "행정동명",
    "lctnAddr",
    "zoneArea",
    "land_type_clean",
    "대표추천유형",
    "추천가능유형",
    "추천가능유형수",
    "추천근거",

    "최고_수요균형도",
    "면적_pct",
    "면적_상위비율",

    "20대미만_활동인구",
    "20대미만_구성비",
    "20대미만_LQ",

    "60대이상_활동인구",
    "60대이상_구성비",
    "60대이상_LQ",

    "전체활동인구",
    "직장_LQ",
    "방문_LQ",

    "아동돌봄시설_전체",
    "고령층생활서비스_전체",

    "아동_우선검토균형도",
    "고령_우선검토균형도",
    "활동교류_우선검토균형도",

    "x",
    "y",
    "검토유의사항"
]

final_output = final_candidates[
    final_output_cols
].copy()

print("최종 분석 테이블:", final_output.shape)
print("고유 주소:", final_output["lctnAddr"].nunique())
print("결측 주소:", final_output["lctnAddr"].isna().sum())

display(final_output.head())

In [ ]:
final_output.to_csv(
    BASE_PATH
    + "03_analysis_output/최종_유휴공유재산_우선검토대상_186건.csv",
    index=False,
    encoding="utf-8-sig"
)

print("최종 분석 테이블 저장 완료")

# ✅ 최종 분석 결과

지역수요와 유휴 공유재산의 면적·지목 조건을 결합하여
총 186건의 1차 우선 검토 대상을 도출하였다.

대표 추천유형은 다음과 같다.

- 고령 생활지원형: 96건
- 아동·가족 생활지원형: 73건
- 활동·교류 지원형: 17건

각 재산에는 단일 활용유형만 강제하지 않고
복수의 추천 가능 유형을 함께 보존하였다.

최종 결과는 실제 사업 시행 대상을 확정하는 것이 아니라,
지역 생활수요와 현재 확보 가능한 재산정보를 기준으로
향후 법적·물리적·사업성 검토가 우선적으로 필요한 대상을
선별한 결과이다.

In [ ]:
import pandas as pd
import numpy as np

BASE = "/content/drive/MyDrive/부산_BigData_공모전/"

# 최종 186건
final_186 = pd.read_csv(
    BASE + "03_analysis_output/최종_유휴공유재산_우선검토대상_186건.csv"
)

# 지역수요 프로필
profile = pd.read_csv(
    BASE + "02_processed_data/지역수요프로필_행정동별.csv"
)

print("최종 후보:", final_186.shape)
print("지역수요 프로필:", profile.shape)

print("\n[지역수요 프로필 컬럼]")
print(profile.columns.tolist())

In [ ]:
import pandas as pd
import numpy as np

BASE = "/content/drive/MyDrive/부산_BigData_공모전/"

# 저장된 결과 불러오기
final_186 = pd.read_csv(
    BASE + "03_analysis_output/최종_유휴공유재산_우선검토대상_186건.csv"
)

profile = pd.read_csv(
    BASE + "02_processed_data/지역수요프로필_행정동별.csv"
)

# 부산 전체 기준 구성비
city_under20_share = (
    profile["20대미만_활동인구"].sum()
    / profile["연령합계_전체활동인구"].sum()
)

city_over60_share = (
    profile["60대이상_활동인구"].sum()
    / profile["연령합계_전체활동인구"].sum()
)

city_work_share = (
    profile["평균직장인구수"].sum()
    / profile["전체활동인구"].sum()
)

city_visit_share = (
    profile["평균방문인구수"].sum()
    / profile["전체활동인구"].sum()
)

# LQ 재계산
profile["20대미만_LQ"] = (
    profile["20대미만_구성비"] / city_under20_share
)

profile["60대이상_LQ"] = (
    profile["60대이상_구성비"] / city_over60_share
)

profile["직장_LQ"] = (
    profile["직장인구_구성비"] / city_work_share
)

profile["방문_LQ"] = (
    profile["방문인구_구성비"] / city_visit_share
)

profile["활동교류_LQ"] = profile[
    ["직장_LQ", "방문_LQ"]
].max(axis=1)

# 206개 행정동 기준 백분위
profile["아동_규모_pct"] = (
    profile["20대미만_활동인구"].rank(pct=True)
)

profile["아동_구조_pct"] = (
    profile["20대미만_LQ"].rank(pct=True)
)

profile["고령_규모_pct"] = (
    profile["60대이상_활동인구"].rank(pct=True)
)

profile["고령_구조_pct"] = (
    profile["60대이상_LQ"].rank(pct=True)
)

profile["활동규모_pct"] = (
    profile["전체활동인구"].rank(pct=True)
)

profile["활동교류_특화_pct"] = (
    profile["활동교류_LQ"].rank(pct=True)
)

# 수요균형도 재계산
profile["아동_수요균형도"] = profile[
    ["아동_규모_pct", "아동_구조_pct"]
].min(axis=1)

profile["고령_수요균형도"] = profile[
    ["고령_규모_pct", "고령_구조_pct"]
].min(axis=1)

profile["활동교류_수요균형도"] = profile[
    ["활동규모_pct", "활동교류_특화_pct"]
].min(axis=1)

# 최종 후보에 지역 수요균형도 결합
check = final_186.merge(
    profile[
        [
            "행정동명",
            "아동_수요균형도",
            "고령_수요균형도",
            "활동교류_수요균형도"
        ]
    ],
    on="행정동명",
    how="left",
    validate="many_to_one"
)

type_score = {
    "아동·가족 생활지원형": "아동_수요균형도",
    "고령 생활지원형": "고령_수요균형도",
    "활동·교류 지원형": "활동교류_수요균형도"
}

# 실제 후보유형에 해당하는 점수만 남기기
for type_name, score_col in type_score.items():
    check[type_name] = np.where(
        check["추천가능유형"].str.contains(
            type_name,
            regex=False,
            na=False
        ),
        check[score_col],
        np.nan
    )

type_cols = list(type_score.keys())

# 후보유형 중 최고점 계산
check["재계산_최고점"] = check[type_cols].max(axis=1)

# 최고점과 정확히 같은 후보유형 수
check["최고점_유형수"] = check[type_cols].eq(
    check["재계산_최고점"],
    axis=0
).sum(axis=1)

ties = check[
    check["최고점_유형수"] >= 2
].copy()

# 대표추천유형이 재계산 결과와 일치하는지도 확인
check["재계산_대표유형"] = check[type_cols].idxmax(axis=1)

mismatch = check[
    check["대표추천유형"] != check["재계산_대표유형"]
].copy()

print("===== 검증 결과 =====")
print("최종 후보 건수:", len(check))
print("지역수요 결합 실패:", check["아동_수요균형도"].isna().sum())
print("최고 수요균형도 동점 재산 수:", len(ties))
print("기존 대표추천유형과 재계산 결과 불일치:", len(mismatch))

print("\n[대표추천유형 건수]")
print(check["재계산_대표유형"].value_counts())

if len(ties) > 0:
    print("\n===== 동점 재산 =====")
    display(
        ties[
            [
                "행정동명",
                "lctnAddr",
                "추천가능유형",
                "아동·가족 생활지원형",
                "고령 생활지원형",
                "활동·교류 지원형",
                "대표추천유형",
                "재계산_대표유형"
            ]
        ]
    )

if len(mismatch) > 0:
    print("\n===== 대표유형 불일치 재산 =====")
    display(
        mismatch[
            [
                "행정동명",
                "lctnAddr",
                "추천가능유형",
                "대표추천유형",
                "재계산_대표유형"
            ]
        ]
    )